# Mise en oeuvre de la bibliothèque PySHS

Cette bibliothèque permet de faciliter le traitement de données tabulaire de sondages/questionnaires

## Chargement de la bibliothèque

Chargement de la bibliothèque

In [1]:
import sys
sys.path.append('..')
import pyshs
import pandas as pd

## Chargement des données de test

Chargement de données d'une enquête passée en 2011 avec la bibliothèque `Pandas`

> Daniel, Boy. 2012. Les Représentations Sociales de La Science et de La Technique, format SPSS 

In [2]:
import pandas as pd
data = pd.read_spss('bdd2011.sav')
data.head()

,ident,poids1,in06,regsof,habp,rs1,rs2,age,rs3,rs5,...,intedom,infonuc,imache,RecQ7,RecQ8,recQ9,Q7Q8Q9,typonuc,risque,fxdate
0,2.0,1.113456,Meurthe-et-Moselle,Est,Plus de 100 000 habitants,Femme,23,18 à 24 ans,J'ai un travail,Salarié d'une entreprise privée,...,Cinq domaines,Plutôt mal informé,Négatif,1.0,1.0,3.0,113.0,Pour et hésite,Trois risques,2017-11-16T14:13
1,3.0,0.752896,Meurthe-et-Moselle,Est,Plus de 100 000 habitants,Homme,41,35 à 49 ans,J'ai un travail,Salarié de l'Etat ou d'une collectivité locale,...,Aucun domaine d'intérêt,Très ou plutôt bien informé,Positif,1.0,1.0,1.0,111.0,Pour,Un risque,2017-11-16T15:00
2,4.0,0.794269,Meurthe-et-Moselle,Est,Plus de 100 000 habitants,Femme,63,50 à 64 ans,Je suis à la retraite ou pré-retraite,Salarié de l'Etat ou d'une collectivité locale,...,Neuf domaines,Plutôt mal informé,Ambigu,1.0,1.0,3.0,113.0,Pour et hésite,Deux risques,2017-11-16T15:31
3,5.0,0.848631,Meurthe-et-Moselle,Est,Plus de 100 000 habitants,Homme,30,25 à 34 ans,J'ai un travail,Salarié d'une entreprise privée,...,Six domaines,Très ou plutôt bien informé,Positif,1.0,2.0,3.0,123.0,Hésitants,Aucun risque,2017-11-16T15:42
4,6.0,1.020218,Meurthe-et-Moselle,Est,Plus de 100 000 habitants,Homme,70,65 ans et+,Je suis à la retraite ou pré-retraite,Salarié de l'Etat ou d'une collectivité locale,...,Deux domaines,Plutôt mal informé,Positif,1.0,1.0,1.0,111.0,Pour,Trois risques,2017-11-16T16:15


Recodage des variables (utilisation de la méthode replace de Pandas)

## Étape de recodage

En s'appuyant sur la bibliothèque Pandas

In [3]:
# Créer trois nouvelles variables recodées suivant ce que l'on souhaite

data["genre"] = data["rs1"].astype("O")

reco = {'Assez':"2 - Assez", 'Peu':"1 - Peu ou pas du tout", 
        'Pas du tout':"1 - Peu ou pas du tout", 'Beaucoup':"3 - Beaucoup"}
data["interetscience"] = data['q2'].astype("O").replace(reco).fillna("4 - Sans réponse")

reco = {'A peu près autant de bien que de mal':"2 - Autant de bien que de mal",
        'Plus de mal que de bien':"1 - Plus de mal que de bien", 
         'Plus de bien que de mal':"3 - Plus de bien que de mal"}
data["apportscience"] = data['q15'].astype("O").replace(reco).fillna("4 - Sans réponse")

# Variable dichotomisée 1/0 sur l'apport positif de la science
data["apportsciencepositif"] = data["q15"].apply(lambda x : 1 if x == "Plus de bien que de mal" else 0).fillna(0)

### Description de variables

In [4]:
pyshs.description(data[["age","genre","interetscience","apportscience","RecQ7"]])

,Type,Modalités,Mode,Moyenne,Écart-type,Valeurs manquantes
Variable,,,,,,
age,Catégorielle,5.0,35 à 49 ans,,,0
genre,Catégorielle,2.0,Femme,,,0
interetscience,Catégorielle,4.0,1 - Peu ou pas du tout,,,0
apportscience,Catégorielle,4.0,2 - Autant de bien que de mal,,,0
RecQ7,Numérique,,,1.78,0.79,0


### Vérification d'un recodage

**Attention** Il n'y a pas de rendu sur Github du diagramme (qui repose sur plotly)

In [5]:
pyshs.verification_recodage(data,"apportscience","apportsciencepositif")

### Tri à plat pondéré

In [7]:
pyshs.tri_a_plat(data,"interetscience","poids1", arrondir=1)

,Effectif,Pourcentage (%)
1 - Peu ou pas du tout,462.1,45.0
2 - Assez,413.3,40.2
3 - Beaucoup,150.6,14.7
4 - Sans réponse,1.0,0.1
Total,1027.0,100.0


### Tableau croisé pondéré

In [9]:
pyshs.tableau_croise(data,"genre","interetscience","poids1", arrondir=2)

interetscience,1 - Peu ou pas du tout,2 - Assez,3 - Beaucoup,4 - Sans réponse,Total
genre,,,,,
Femme,272.24 (50.59%),205.1 (38.12%),60.74 (11.29%),0.0 (0.0%),538.08 (100%)
Homme,189.81 (38.82%),208.23 (42.59%),89.91 (18.39%),0.97 (0.2%),488.92 (100%)
Total,462.05 (44.99%),413.33 (40.25%),150.65 (14.67%),0.97 (0.09%),1027.0 (100%)


### Tableau croisé contrôlant une variable

In [4]:
pyshs.tableau_croise_controle(data,"genre","interetscience","apportscience","poids1",chi2=True, arrondir=1, proba_simplifiee=False)

/opt/anaconda3/lib/python3.11/site-packages/samplics/categorical/tabulation.py:641: UserWarning: Matrix is computationally singular, results may be inaccurate
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/samplics/categorical/tabulation.py:641: UserWarning: Matrix is computationally singular, results may be inaccurate
  warnings.warn(


apportscience                              1 - Plus de mal que de bien  \
genre               interetscience                                       
Femme (p = 1.1e-01) 1 - Peu ou pas du tout                 13.4 (4.9%)   
                    2 - Assez                              13.9 (6.8%)   
                    3 - Beaucoup                            2.8 (4.6%)   
                    Total                                  30.1 (5.6%)   
Homme (p = 1.1e-06) 1 - Peu ou pas du tout                 12.3 (6.5%)   
                    2 - Assez                              11.4 (5.5%)   
                    3 - Beaucoup                            5.6 (6.2%)   
                    4 - Sans réponse                      1.0 (100.0%)   
                    Total                                  30.3 (6.2%)   

apportscience                              2 - Autant de bien que de mal  \
genre               interetscience                                         
Femme (p = 1.1e-01) 1 - Peu ou pas du tout                 166.0 (61.0%)   
                    2 - Assez                              106.1 (51.7%)   
                    3 - Beaucoup                            28.9 (47.6%)   
                    Total                                  301.0 (55.9%)   
Homme (p = 1.1e-06) 1 - Peu ou pas du tout                 109.3 (57.6%)   
                    2 - Assez                               91.7 (44.0%)   
                    3 - Beaucoup                            29.7 (33.0%)   
                    4 - Sans réponse                          0.0 (0.0%)   
                    Total                                  230.7 (47.2%)   

apportscience                              3 - Plus de bien que de mal  \
genre               interetscience                                       
Femme (p = 1.1e-01) 1 - Peu ou pas du tout                86.5 (31.8%)   
                    2 - Assez                             80.8 (39.4%)   
                    3 - Beaucoup                          29.0 (47.8%)   
                    Total                                196.3 (36.5%)   
Homme (p = 1.1e-06) 1 - Peu ou pas du tout                65.5 (34.5%)   
                    2 - Assez                            105.2 (50.5%)   
                    3 - Beaucoup                          53.8 (59.8%)   
                    4 - Sans réponse                        0.0 (0.0%)   
                    Total                                224.4 (45.9%)   

apportscience                              4 - Sans réponse         Total  
genre               interetscience                                         
Femme (p = 1.1e-01) 1 - Peu ou pas du tout       6.4 (2.4%)  272.2 (100%)  
                    2 - Assez                    4.3 (2.1%)  205.1 (100%)  
                    3 - Beaucoup                 0.0 (0.0%)   60.7 (100%)  
                    Total                       10.7 (2.0%)  538.1 (100%)  
Homme (p = 1.1e-06) 1 - Peu ou pas du tout       2.7 (1.4%)  189.8 (100%)  
                    2 - Assez                    0.0 (0.0%)  208.2 (100%)  
                    3 - Beaucoup                 0.8 (0.9%)   89.9 (100%)  
                    4 - Sans réponse             0.0 (0.0%)    1.0 (100%)  
                    Total                        3.5 (0.7%)  488.9 (100%)

### Tableau croisé multiple pondéré

In [5]:
pyshs.tableau_croise_multiple(data,"apportscience",
                                {"genre":"Genre",
                                 "interetscience":"Intérêt Science"},
                                "poids1",axis=0, arrondir=1, proba_simplifiee=True)

/opt/anaconda3/lib/python3.11/site-packages/samplics/categorical/tabulation.py:641: UserWarning: Matrix is computationally singular, results may be inaccurate
  warnings.warn(


1 - Plus de mal que de bien  \
Variable                        Modalités                                            
Genre *** (p < 0.001)           Femme                                  30.1 (5.6%)   
                                Homme                                  30.3 (6.2%)   
                                Total                                  60.4 (5.9%)   
Intérêt Science *** (p < 0.001) 1 - Peu ou pas du tout                 25.7 (5.6%)   
                                2 - Assez                              25.3 (6.1%)   
                                3 - Beaucoup                            8.4 (5.6%)   
                                4 - Sans réponse                      1.0 (100.0%)   
                                Total                                  60.4 (5.9%)   

                                                       2 - Autant de bien que de mal  \
Variable                        Modalités                                              
Genre *** (p < 0.001)           Femme                                  301.0 (55.9%)   
                                Homme                                  230.7 (47.2%)   
                                Total                                  531.7 (51.8%)   
Intérêt Science *** (p < 0.001) 1 - Peu ou pas du tout                 275.3 (59.6%)   
                                2 - Assez                              197.8 (47.9%)   
                                3 - Beaucoup                            58.6 (38.9%)   
                                4 - Sans réponse                          0.0 (0.0%)   
                                Total                                  531.7 (51.8%)   

                                                       3 - Plus de bien que de mal  \
Variable                        Modalités                                            
Genre *** (p < 0.001)           Femme                                196.3 (36.5%)   
                                Homme                                224.4 (45.9%)   
                                Total                                420.7 (41.0%)   
Intérêt Science *** (p < 0.001) 1 - Peu ou pas du tout               152.0 (32.9%)   
                                2 - Assez                            185.9 (45.0%)   
                                3 - Beaucoup                          82.8 (55.0%)   
                                4 - Sans réponse                        0.0 (0.0%)   
                                Total                                420.7 (41.0%)   

                                                       4 - Sans réponse  \
Variable                        Modalités                                 
Genre *** (p < 0.001)           Femme                       10.7 (2.0%)   
                                Homme                        3.5 (0.7%)   
                                Total                       14.2 (1.4%)   
Intérêt Science *** (p < 0.001) 1 - Peu ou pas du tout       9.1 (2.0%)   
                                2 - Assez                    4.3 (1.0%)   
                                3 - Beaucoup                 0.8 (0.5%)   
                                4 - Sans réponse             0.0 (0.0%)   
                                Total                       14.2 (1.4%)   

                                                                Total  \
Variable                        Modalités                               
Genre *** (p < 0.001)           Femme                    538.1 (100%)   
                                Homme                    488.9 (100%)   
                                Total                   1027.0 (100%)   
Intérêt Science *** (p < 0.001) 1 - Peu ou pas du tout   462.1 (100%)   
                                2 - Assez                413.3 (100%)   
                                3 - Beaucoup             150.6 (100%)   
                                4 - Sans réponse           1.0 (100%)   
                                Total       

### Régression logistique

In [6]:
reg = pyshs.regression_logistique(data,"apportsciencepositif",
                    {"genre":"Genre",
                     "age":"Age"},
                     "poids1", arrondir=2, notationscientifique=False, sig=True)
reg

OR                p            IC 95%
Variable   Modalité                                            
.Intercept              0.41  *** (p < 0.001)  0.41 [0.27-0.62]
Age        18 à 24 ans   ref                                   
           25 à 34 ans  1.12         (p=0.67)  1.12 [0.67-1.87]
           35 à 49 ans  1.36          (p=0.2)  1.36 [0.85-2.17]
           50 à 64 ans  1.73     * (p < 0.05)  1.73 [1.09-2.77]
           65 ans et+   1.74     * (p < 0.05)  1.74 [1.08-2.79]
Genre      Femme         ref                                   
           Homme        1.41     * (p < 0.05)   1.41 [1.1-1.82]

### Rassembler les tableaux et les mettre dans un fichier excel

In [12]:
# Définir un dictionnaire titre:tableau 
tableaux = {
    "Tri à plat de interetscience":pyshs.tri_a_plat(data,"interetscience","poids1"),
    "Tableau croisé entre interetscience et genre":pyshs.tableau_croise(data,"genre","interetscience","poids1"),
    "Régression logistique apport science positif":reg
    }

# Ecrire dans le fichier excel
pyshs.vers_excel(tableaux,"resultats.xlsx")

### Association entre une variable catégorielle et les autres variables

Implémentation de la variable `catdes` de la bibliothèque R FactoMineR

In [27]:
var_quali, var_num, mod_quali, mod_num = pyshs.catdes(data,
                                                vardep = "apportscience",
                                                varindep = ["genre","interetscience","rs2"],
                                                weight="poids1",
                                                mod = True)

In [28]:
var_quali

,p,df
apportscience,,
interetscience,5.759929e-07,9
genre,5.866780e-03,3
rs2,1.431711e-02,210


In [29]:
var_num

,Eta 2,p-value
apportscience,,


In [30]:
mod_quali

Cla/Mod (n_kj/n_j)  \
                                            var                                                         
apportscience_1 - Plus de mal que de bien   rs2_34                                              37.50   
                                            interetscience_4 - Sans réponse                    100.00   
                                            rs2_32                                              17.65   
apportscience_2 - Autant de bien que de mal interetscience_1 - Peu ou pas du tout               59.52   
                                            interetscience_3 - Beaucoup                         39.07   
                                            rs2_79                                               9.09   
                                            genre_Femme                                         55.95   
                                            genre_Homme                                         47.24   
                                            rs2_36                                              76.19   
                                            rs2_66                                              12.50   
                                            rs2_63                                              30.43   
                                            interetscience_2 - Assez                            47.94   
                                            rs2_27                                              76.47   
                                            rs2_29                                              81.82   
                                            rs2_32                                              29.41   
                                            rs2_53                                              80.00   
apportscience_3 - Plus de bien que de mal   interetscience_1 - Peu ou pas du tout               32.90   
                                            interetscience_3 - Beaucoup                         54.97   
                                            rs2_79                                              90.91   
                                            genre_Femme                                         36.43   
                                            genre_Homme                                         45.81   
                                            rs2_29                                               0.00   
                                            rs2_55                                              72.22   
                                            rs2_66                                              87.50   
                                            rs2_27                                              11.76   
                                            rs2_63                                              65.22   
                                            interetscience_2 - Assez                            45.04   
                                            rs2_71                                              66.67   
                                            rs2_54                                              62.50   
apportscience_4 - Sans réponse              rs2_19                                              16.67   
                                            genre_Femme                                          2.04   
                                            genre_Homme                                          0.61   

                                                                                   Mod/Cla (n_kj/n_k)  \
                                            var                                                         
apportscience_1 - Plus de mal que de bien   rs2_34                                              10.00   
                                            interetscience_4 - Sans réponse                      1.67   
                                            rs2_32                                          

In [31]:
mod_num

,,Valeur test,p-value,Moy mod,Moy glob,Std mod,Std glob
,var,,,,,,
